# Emotion CNN Training (RAVDESS + CREMA-D + TESS)

Trains an 8-class emotion classifier from speech audio using MFCC features.
Exports to ONNX and uploads directly to GCS bucket `redline-ai-models`.

**Runtime → Change runtime type → T4 GPU** for faster training.

Steps:
1. Install dependencies
2. Download datasets (RAVDESS, CREMA-D, TESS)
3. Train 4-layer CNN
4. Export to ONNX
5. Upload to GCS & restart Cloud Run

In [ ]:
!pip -q install torch torchaudio soundfile scikit-learn onnx onnxruntime google-cloud-storage kaggle gdown

## 1. Download Datasets

The cell below will ask you to paste your Kaggle API token. You already have it — just paste it when prompted.

In [ ]:
import os
import subprocess
import zipfile
import urllib.request
from pathlib import Path
from getpass import getpass

DATASET_ROOT = Path("/content/datasets")
DATASET_ROOT.mkdir(exist_ok=True)

# ── Kaggle auth via API token ────────────────────────────────────
if not os.environ.get("KAGGLE_API_TOKEN"):
    token = getpass("Paste your Kaggle API token: ")
    os.environ["KAGGLE_API_TOKEN"] = token.strip()
    print("Kaggle token set.")
else:
    print("Kaggle token already set.")

# ── RAVDESS (Zenodo — stable) ────────────────────────────────────
ravdess_dir = DATASET_ROOT / "ravdess"
ravdess_zip = DATASET_ROOT / "ravdess.zip"
print("\n[RAVDESS]")
if ravdess_dir.exists() and any(ravdess_dir.rglob("*.wav")):
    print(f"  Already exists ({len(list(ravdess_dir.rglob('*.wav')))} wav files)")
else:
    if not ravdess_zip.exists():
        print("  Downloading from Zenodo ...")
        urllib.request.urlretrieve(
            "https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip?download=1",
            str(ravdess_zip),
        )
        print(f"  Downloaded ({ravdess_zip.stat().st_size / 1024 / 1024:.1f} MB)")
    ravdess_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ravdess_zip, "r") as zf:
        zf.extractall(ravdess_dir)
    print(f"  Done: {len(list(ravdess_dir.rglob('*.wav')))} wav files")

# ── TESS (Kaggle) ────────────────────────────────────────────────
tess_dir = DATASET_ROOT / "tess"
print("\n[TESS]")
if tess_dir.exists() and any(tess_dir.rglob("*.wav")):
    print(f"  Already exists ({len(list(tess_dir.rglob('*.wav')))} wav files)")
else:
    print("  Downloading from Kaggle ...")
    subprocess.run([
        "kaggle", "datasets", "download", "-d",
        "ejlok1/toronto-emotional-speech-set-tess",
        "-p", str(DATASET_ROOT),
    ], check=True)
    tess_zip = DATASET_ROOT / "toronto-emotional-speech-set-tess.zip"
    tess_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(tess_zip, "r") as zf:
        zf.extractall(tess_dir)
    print(f"  Done: {len(list(tess_dir.rglob('*.wav')))} wav files")

# ── CREMA-D (Kaggle) ─────────────────────────────────────────────
cremad_dir = DATASET_ROOT / "crema-d"
print("\n[CREMA-D]")
if cremad_dir.exists() and any(cremad_dir.rglob("*.wav")):
    print(f"  Already exists ({len(list(cremad_dir.rglob('*.wav')))} wav files)")
else:
    print("  Downloading from Kaggle ...")
    subprocess.run([
        "kaggle", "datasets", "download", "-d",
        "ejlok1/cremad",
        "-p", str(DATASET_ROOT),
    ], check=True)
    cremad_zip = DATASET_ROOT / "cremad.zip"
    cremad_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(cremad_zip, "r") as zf:
        zf.extractall(cremad_dir)
    print(f"  Done: {len(list(cremad_dir.rglob('*.wav')))} wav files")

print("\n✅ All datasets ready.")

## 2. Dataset Scanning & Preparation

In [ ]:
import random
import math
from dataclasses import dataclass
from typing import Optional

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio.transforms as T
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset, Subset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

EMOTION_TO_ID = {
    "neutral": 0, "calm": 1, "happy": 2, "sad": 3,
    "angry": 4, "fearful": 5, "disgust": 6, "surprised": 7,
}
ID_TO_EMOTION = {v: k for k, v in EMOTION_TO_ID.items()}


@dataclass(frozen=True)
class AudioSample:
    path: str
    label: int
    source: str


def scan_ravdess(root: Path):
    mapping = {"01": "neutral", "02": "calm", "03": "happy", "04": "sad",
               "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised"}
    out = []
    for wav in root.rglob("*.wav"):
        parts = wav.name.split("-")
        if len(parts) >= 3 and parts[2] in mapping:
            out.append(AudioSample(str(wav), EMOTION_TO_ID[mapping[parts[2]]], "ravdess"))
    return out


def scan_cremad(root: Path):
    mapping = {"ANG": "angry", "DIS": "disgust", "FEA": "fearful",
               "HAP": "happy", "NEU": "neutral", "SAD": "sad"}
    out = []
    for wav in root.rglob("*.wav"):
        parts = wav.stem.split("_")
        if len(parts) >= 3:
            code = parts[2].upper()
            if code in mapping:
                out.append(AudioSample(str(wav), EMOTION_TO_ID[mapping[code]], "crema_d"))
    return out


def scan_tess(root: Path):
    mapping = {"angry": "angry", "disgust": "disgust", "fear": "fearful",
               "happy": "happy", "neutral": "neutral", "sad": "sad", "ps": "surprised"}
    out = []
    for wav in root.rglob("*.wav"):
        lower = wav.stem.lower()
        emotion = None
        if "_" in lower:
            token = lower.split("_")[-1]
            emotion = mapping.get(token)
        if emotion is None:
            for key, val in mapping.items():
                if key in lower:
                    emotion = val
                    break
        if emotion:
            out.append(AudioSample(str(wav), EMOTION_TO_ID[emotion], "tess"))
    return out


# Collect all samples
samples = []
samples.extend(scan_ravdess(DATASET_ROOT / "ravdess"))
samples.extend(scan_cremad(DATASET_ROOT / "crema-d"))
samples.extend(scan_tess(DATASET_ROOT / "tess"))

labels = [s.label for s in samples]
unique, counts = np.unique(labels, return_counts=True)
print(f"Total samples: {len(samples)}")
print("Class distribution:")
for u, c in zip(unique, counts):
    print(f"  {ID_TO_EMOTION[u]:12s}: {c}")

## 3. MFCC Dataset & DataLoaders

In [ ]:
class EmotionMFCCDataset(Dataset):
    def __init__(self, samples, sample_rate=16000, duration=3, n_mfcc=40, max_t=94,
                 mean=None, std=None):
        self.samples = samples
        self.sr = sample_rate
        self.max_len = sample_rate * duration
        self.max_t = max_t
        self.mean = mean
        self.std = std
        self.mfcc_transform = T.MFCC(
            sample_rate=sample_rate, n_mfcc=n_mfcc,
            melkwargs={"n_fft": 1024, "hop_length": 512, "n_mels": 64},
        )

    def set_norm(self, mean, std):
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        wav, sr = sf.read(s.path)
        audio = torch.tensor(wav, dtype=torch.float32)
        if audio.ndim > 1:
            audio = audio.mean(dim=1)
        audio = audio.unsqueeze(0)
        if sr != self.sr:
            audio = T.Resample(sr, self.sr)(audio)
        if audio.shape[1] > self.max_len:
            audio = audio[:, :self.max_len]
        elif audio.shape[1] < self.max_len:
            audio = torch.nn.functional.pad(audio, (0, self.max_len - audio.shape[1]))
        mfcc = self.mfcc_transform(audio)
        if mfcc.shape[2] > self.max_t:
            mfcc = mfcc[:, :, :self.max_t]
        elif mfcc.shape[2] < self.max_t:
            mfcc = torch.nn.functional.pad(mfcc, (0, self.max_t - mfcc.shape[2]))
        if self.mean is not None and self.std is not None:
            denom = self.std if self.std > 1e-8 else 1.0
            mfcc = (mfcc - self.mean) / denom
        return mfcc, torch.tensor(s.label, dtype=torch.long)


# Split: 80% train, 10% val, 10% test (stratified)
labels_arr = np.array(labels)
indices = np.arange(len(samples))

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, holdout_idx = next(sss1.split(indices, labels_arr))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_rel, test_rel = next(sss2.split(holdout_idx, labels_arr[holdout_idx]))
val_idx = holdout_idx[val_rel]
test_idx = holdout_idx[test_rel]

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

full_ds = EmotionMFCCDataset(samples)
train_ds = Subset(full_ds, train_idx.tolist())
val_ds = Subset(full_ds, val_idx.tolist())
test_ds = Subset(full_ds, test_idx.tolist())

# Compute MFCC normalization stats from training set
print("Computing MFCC normalization stats ...")
loader_stats = DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=0)
total_sum, total_sq, total_n = 0.0, 0.0, 0
for x, _ in loader_stats:
    total_sum += x.sum().item()
    total_sq += (x * x).sum().item()
    total_n += x.numel()
mfcc_mean = total_sum / total_n
mfcc_std = math.sqrt(max((total_sq / total_n) - mfcc_mean**2, 1e-12))
full_ds.set_norm(mfcc_mean, mfcc_std)
print(f"MFCC mean={mfcc_mean:.4f}, std={mfcc_std:.4f}")

BATCH_SIZE = 32
pin = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)
print("DataLoaders ready.")

## 4. Model Definition

In [ ]:
class EmotionModel(nn.Module):
    """Same architecture as ml/model.py — must match for ONNX compatibility."""
    def __init__(self, num_classes=8):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self._init_fc(num_classes)

    def _init_fc(self, num_classes):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 40, 94)
            out = self.conv(dummy)
            self.flat_size = out.view(1, -1).size(1)
        self.fc = nn.Sequential(
            nn.Linear(self.flat_size, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EmotionModel(num_classes=8).to(device)
print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training Loop

In [ ]:
from torch.optim import AdamW

EPOCHS = 80
LR = 3e-4
PATIENCE = 12
LABEL_SMOOTHING = 0.1

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)


def run_epoch(model, loader, criterion, optimizer, device):
    is_train = optimizer is not None
    model.train(is_train)
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        if is_train:
            loss.backward()
            optimizer.step()
        loss_sum += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total


best_val_acc = -1.0
best_epoch = -1
patience_counter = 0
best_path = "/content/emotion_model_best.pt"

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, None, device)
    scheduler.step(val_acc)

    print(f"Epoch {epoch:03d} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc*100:.2f}% | "
          f"val_loss={val_loss:.4f} val_acc={val_acc*100:.2f}% | "
          f"lr={optimizer.param_groups[0]['lr']:.2e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), best_path)
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

print(f"\nBest epoch: {best_epoch}, Best val accuracy: {best_val_acc*100:.2f}%")

## 6. Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import json

# Load best model
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(y.tolist())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"Test accuracy: {test_acc*100:.2f}%\n")

target_names = [ID_TO_EMOTION[i] for i in range(8)]
# Only include labels that appear in the test set
present_labels = sorted(set(all_labels))
present_names = [ID_TO_EMOTION[i] for i in present_labels]
print(classification_report(all_labels, all_preds, labels=present_labels, target_names=present_names))

## 7. Export to ONNX

In [ ]:
import onnx
import onnxruntime as ort

ONNX_PATH = "/content/emotion_model.onnx"

model.cpu().eval()
dummy = torch.zeros(1, 1, 40, 94)

torch.onnx.export(
    model, dummy, ONNX_PATH,
    export_params=True, opset_version=17, do_constant_folding=True,
    input_names=["mfcc"], output_names=["logits"],
    dynamic_axes={"mfcc": {0: "batch_size"}, "logits": {0: "batch_size"}},
)

# Validate
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
ort_out = session.run(None, {"mfcc": dummy.numpy()})
print(f"ONNX export: {ONNX_PATH}")
print(f"Output shape: {np.array(ort_out[0]).shape}")
print(f"File size: {os.path.getsize(ONNX_PATH) / 1024 / 1024:.1f} MB")

# Verify ONNX matches PyTorch
with torch.no_grad():
    pt_out = model(dummy).numpy()
diff = np.abs(pt_out - ort_out[0]).max()
print(f"Max PyTorch vs ONNX difference: {diff:.8f} (should be < 1e-5)")

## 8. Save Training Metadata

In [ ]:
metadata = {
    "seed": SEED,
    "num_samples": len(samples),
    "train_size": len(train_idx),
    "val_size": len(val_idx),
    "test_size": len(test_idx),
    "mfcc_mean": mfcc_mean,
    "mfcc_std": mfcc_std,
    "best_epoch": best_epoch,
    "best_val_accuracy": best_val_acc,
    "test_accuracy": test_acc,
    "epochs_config": EPOCHS,
    "lr": LR,
    "label_smoothing": LABEL_SMOOTHING,
    "patience": PATIENCE,
    "label_map": ID_TO_EMOTION,
}

with open("/content/emotion_training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Also save the .pt checkpoint
torch.save(model.state_dict(), "/content/emotion_model.pt")

print("Saved:")
print("  /content/emotion_model.onnx")
print("  /content/emotion_model.pt")
print("  /content/emotion_training_metadata.json")

## 9. Upload to GCS

Authenticate with your GCP project, then upload the trained ONNX model to the bucket.

In [ ]:
# Authenticate to GCP (will open a popup)
from google.colab import auth
auth.authenticate_user()
print("Authenticated.")

In [ ]:
from google.cloud import storage

PROJECT_ID = "redlineai-490103"
BUCKET_NAME = "redline-ai-models"
GCS_BLOB = "models/emotion_model.onnx"

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

# Upload ONNX model
blob = bucket.blob(GCS_BLOB)
blob.upload_from_filename(ONNX_PATH)
print(f"Uploaded {ONNX_PATH} -> gs://{BUCKET_NAME}/{GCS_BLOB}")

# Also upload the .pt checkpoint as backup
blob_pt = bucket.blob("models/emotion_model.pt")
blob_pt.upload_from_filename("/content/emotion_model.pt")
print(f"Uploaded emotion_model.pt -> gs://{BUCKET_NAME}/models/emotion_model.pt")

# Upload metadata
blob_meta = bucket.blob("models/emotion_training_metadata.json")
blob_meta.upload_from_filename("/content/emotion_training_metadata.json")
print(f"Uploaded metadata -> gs://{BUCKET_NAME}/models/emotion_training_metadata.json")

print(f"\nAll artifacts uploaded to gs://{BUCKET_NAME}/models/")

## 10. Restart Cloud Run Service

Force a new revision so the service downloads the trained model from GCS.

In [ ]:
!gcloud config set project redlineai-490103
!gcloud run services update redline-ai --region=us-central1 --update-env-vars="MODEL_VERSION=$(date +%s)"
print("\nCloud Run service restarted. New revision will download the trained model from GCS.")

In [ ]:
# Verify health after ~2 minutes
import time
import urllib.request
import json

print("Waiting 120s for startup ...")
time.sleep(120)

try:
    url = "https://redline-ai-359883234654.us-central1.run.app/health"
    resp = urllib.request.urlopen(url, timeout=30)
    health = json.loads(resp.read())
    print(json.dumps(health, indent=2))
    if health.get("emotion_model") == "ready":
        print("\n=== Emotion model is READY with trained weights ===")
    else:
        print("\nEmotion model not ready yet — may need more startup time.")
except Exception as e:
    print(f"Health check failed (may still be starting): {e}")
    print("Try again in a few minutes: curl https://redline-ai-359883234654.us-central1.run.app/health")

## Done!

Artifacts:
- **ONNX model**: `gs://redline-ai-models/models/emotion_model.onnx`
- **PyTorch checkpoint**: `gs://redline-ai-models/models/emotion_model.pt`
- **Metadata**: `gs://redline-ai-models/models/emotion_training_metadata.json`

You can download the `.pt` file locally for future retraining:
```python
from google.colab import files
files.download('/content/emotion_model.pt')
files.download('/content/emotion_model.onnx')
```